In [1]:
import json
from pathlib import Path

import datasets as hfds
import pandas as pd
import numpy as np

In [2]:
ROOT = Path("..")

In [3]:
dataset = hfds.load_dataset("clane9/NSD-Flat", download_config=hfds.DownloadConfig(num_proc=8))
dataset = hfds.concatenate_datasets([dataset["train"], dataset["test"]])
print(dataset)

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/44 [00:00<?, ?it/s]

Dataset({
    features: ['subject_id', 'trial_id', 'session_id', 'nsd_id', 'image', 'activity', 'subject', 'flagged', 'BOLD5000', 'shared1000', 'coco_split', 'coco_id', 'objects', 'captions', 'repetitions'],
    num_rows: 213000
})


In [4]:
with (ROOT / "metadata/nsd_include_coco_categories.json").open() as f:
    coco_categories = json.load(f)
    coco_target_id_map = {coco_id: ii for ii, coco_id in enumerate(coco_categories.values())}
    print(coco_categories)
    print(coco_target_id_map)

{'motorcycle': 3, 'airplane': 4, 'bus': 5, 'train': 6, 'fire hydrant': 10, 'stop sign': 11, 'horse': 17, 'sheep': 18, 'cow': 19, 'elephant': 20, 'zebra': 22, 'giraffe': 23, 'umbrella': 25, 'skis': 30, 'snowboard': 31, 'kite': 33, 'skateboard': 36, 'surfboard': 37, 'tennis racket': 38, 'pizza': 53, 'cake': 55, 'bed': 59, 'toilet': 61, 'clock': 74}
{3: 0, 4: 1, 5: 2, 6: 3, 10: 4, 11: 5, 17: 6, 18: 7, 19: 8, 20: 9, 22: 10, 23: 11, 25: 12, 30: 13, 31: 14, 33: 15, 36: 16, 37: 17, 38: 18, 53: 19, 55: 20, 59: 21, 61: 22, 74: 23}


In [5]:
meta_df = pd.read_csv(ROOT / "metadata/nsd_cococlip_metadata.csv")

meta_df["subject_id"] = meta_df["sub"].map({f"subj{ii + 1:02d}": ii for ii in range(8)})
meta_df["session_id"] = meta_df["ses"] - 1
meta_df["target"] = meta_df["target"].map(coco_target_id_map)

print(meta_df.shape)
meta_df.head()

(213000, 11)


,sub,ses,run,trial_id,onset,nsd_id,include,split,target,subject_id,session_id
0,subj01,1,1,0,12.0,46002,False,train,8.0,0,0
1,subj01,1,1,1,16.0,61882,False,train,15.0,0,0
2,subj01,1,1,2,20.0,828,True,train,17.0,0,0
3,subj01,1,1,3,24.0,67573,False,train,17.0,0,0
4,subj01,1,1,4,28.0,16020,False,train,22.0,0,0


In [6]:
index_columns = ["subject_id", "trial_id", "session_id", "nsd_id"]
extra_columns = ["shared1000"]
index_df = dataset.select_columns(index_columns + extra_columns).to_pandas()
print(index_df.shape)
index_df.head()

(213000, 5)


,subject_id,trial_id,session_id,nsd_id,shared1000
0,0,0,0,46002,True
1,0,1,0,61882,False
2,0,2,0,828,False
3,0,3,0,67573,False
4,0,4,0,16020,False


In [7]:
merged_df = index_df.merge(
    meta_df.loc[:, index_columns + ["include", "split", "target"]], on=index_columns
)

# add shared1000 split
# todo: probably should handle this earlier in nsd cococlip generation
assert not (merged_df["shared1000"] & merged_df["include"]).any()
include_labels = np.array(list(coco_categories.values()))
# nb, this doesn't filter out low-confidence samples like we did for other splits
shared1000_mask = merged_df["shared1000"] & np.isin(merged_df["target"], include_labels)
merged_df.loc[shared1000_mask, "include"] = True
merged_df.loc[shared1000_mask, "split"] = "shared1000"

print(merged_df.shape)
merged_df.head()

(213000, 8)


,subject_id,trial_id,session_id,nsd_id,shared1000,include,split,target
0,0,0,0,46002,True,False,train,8.0
1,0,1,0,61882,False,False,train,15.0
2,0,2,0,828,False,True,train,17.0
3,0,3,0,67573,False,False,train,17.0
4,0,4,0,16020,False,False,train,22.0


In [8]:
splits = ["train", "validation", "test", "testid", "shared1000"]

dataset_dict = {}
for split in splits:
    mask = ((merged_df["split"] == split) & merged_df["include"]).values
    split_targets = merged_df.loc[mask, "target"].values
    (split_ids,) = mask.nonzero()
    dataset_dict[split] = (
        dataset.select_columns(["subject_id", "trial_id", "session_id", "nsd_id", "activity"])
        .select(split_ids)
        .add_column("target", split_targets)
    )

dataset_dict = hfds.DatasetDict(dataset_dict)

In [9]:
dataset_dict.save_to_disk(ROOT / "datasets/nsd_flat_cococlip")

Saving the dataset (0/2 shards):   0%|          | 0/32539 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5418 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5390 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5187 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6404 [00:00<?, ? examples/s]

In [10]:
dataset_dict.push_to_hub("clane9/nsd-flat-cococlip")

Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Map:   0%|          | 0/16270 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/6404 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/clane9/nsd-flat-cococlip/commit/8736164c909937b91bd2d41fa282f19715cd9b5c', commit_message='Upload dataset', commit_description='', oid='8736164c909937b91bd2d41fa282f19715cd9b5c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/clane9/nsd-flat-cococlip', endpoint='https://huggingface.co', repo_type='dataset', repo_id='clane9/nsd-flat-cococlip'), pr_revision=None, pr_num=None)